In [9]:
import numpy as np
import pandas as pd
from collections import defaultdict

class LifeInsuranceBayesianModel:
    def __init__(self):
        # Initialize probability tables
        self.prior_prob = None  # P(T)
        self.conditional_probs = defaultdict(dict)  # P(Xi|T)
        
        # Scoring rules from the document
        self.age_rules = [
            (30, 5), (40, 10), (50, 15), (float('inf'), 20)
        ]
        self.health_rules = {
            0: 5,   # Healthy
            1: 10,  # Minor diseases
            2: 15   # Serious/chronic diseases
        }
        self.income_rules = [
            (1700, 15), (3500, 10), (float('inf'), 5)
        ]
        self.dependents_rules = {
            0: 5,   # No dependents
            1: 10,  # 1 dependent
            2: 15,  # 2 dependents
            3: 15    # 3+ dependents
        }
        
    def calculate_score(self, X):
        """Calculate risk score based on the provided rules"""
        score = 0
        
        # Age points
        age = X[1]  # X2 is age
        for threshold, points in self.age_rules:
            if age < threshold:
                score += points
                break
                
        # Health points (chronic diseases)
        health = X[5]  # X6 is chronic diseases
        score += self.health_rules.get(health, 0)
        
        # Income points
        income = X[6]  # X7 is monthly salary (annual not available)
        for threshold, points in self.income_rules:
            if income < threshold:
                score += points
                break
                
        # Dependents points
        dependents = X[3]  # X4 is number of children/dependents
        score += self.dependents_rules.get(min(dependents, 3), 0)
        
        return score
    
    def train(self, data, targets):
        """Train the Bayesian model by calculating probabilities"""
        # Calculate prior probabilities P(T)
        total = len(targets)
        self.prior_prob = {
            0: np.sum(targets == 0) / total,
            1: np.sum(targets == 1) / total
        }
        
        # Calculate conditional probabilities P(Xi|T) for each feature
        features = {
            'Gender': 0,      # X1
            'Age': 1,         # X2 (we'll discretize)
            'Marital': 2,     # X3
            'Dependents': 3,  # X4
            'Activity': 4,    # X5
            'Health': 5,      # X6
            'Income': 6       # X7 (we'll discretize)
        }
        
        # Discretize continuous variables
        age_bins = [0, 30, 40, 50, float('inf')]
        income_bins = [0, 1700, 3500, float('inf')]
        
        for t in [0, 1]:
            t_data = data[targets == t]
            
            # Gender (binary)
            self._calculate_conditional(t, 'Gender', t_data[:, features['Gender']], [0, 1])
            
            # Age (discretized)
            age_discrete = np.digitize(t_data[:, features['Age']], age_bins)
            self._calculate_conditional(t, 'Age', age_discrete, range(1, len(age_bins)+1))
            
            # Marital status (binary)
            self._calculate_conditional(t, 'Marital', t_data[:, features['Marital']], [0, 1])
            
            # Dependents (already discrete)
            self._calculate_conditional(t, 'Dependents', t_data[:, features['Dependents']], [0, 1, 2, 3])
            
            # Physical activity
            self._calculate_conditional(t, 'Activity', t_data[:, features['Activity']], [0, 1, 2])
            
            # Health condition
            self._calculate_conditional(t, 'Health', t_data[:, features['Health']], [0, 1, 2])
            
            # Income (discretized)
            income_discrete = np.digitize(t_data[:, features['Income']], income_bins)
            self._calculate_conditional(t, 'Income', income_discrete, range(1, len(income_bins)+1))
    
    def _calculate_conditional(self, t, feature_name, values, categories):
        """Helper to calculate P(Xi|T) for a feature"""
        total = len(values)
        counts = {cat: np.sum(values == cat) for cat in categories}
        
        # Laplace smoothing to avoid zero probabilities
        smoothed_probs = {
            cat: (count + 1) / (total + len(categories))
            for cat, count in counts.items()
        }
        
        self.conditional_probs[t][feature_name] = smoothed_probs
    
    def predict_probability(self, X):
        """Predict P(T=1|X) using Bayes' theorem"""
        # Calculate P(X|T=1) * P(T=1)
        prob_1 = self._calculate_likelihood(1, X) * self.prior_prob[1]
        
        # Calculate P(X|T=0) * P(T=0)
        prob_0 = self._calculate_likelihood(0, X) * self.prior_prob[0]
        
        # Normalize to get P(T=1|X)
        return prob_1 / (prob_0 + prob_1)
    
    def _calculate_likelihood(self, t, X):
        """Calculate P(X|T=t) by multiplying individual feature probabilities"""
        likelihood = 1.0
        
        # Gender
        gender = X[0]
        likelihood *= self.conditional_probs[t]['Gender'].get(gender, 1e-5)
        
        # Age (discretized)
        age_bins = [0, 30, 40, 50, float('inf')]
        age_cat = np.digitize([X[1]], age_bins)[0]
        likelihood *= self.conditional_probs[t]['Age'].get(age_cat, 1e-5)
        
        # Marital status
        marital = X[2]
        likelihood *= self.conditional_probs[t]['Marital'].get(marital, 1e-5)
        
        # Dependents
        dependents = min(X[3], 3)  # Cap at 3+
        likelihood *= self.conditional_probs[t]['Dependents'].get(dependents, 1e-5)
        
        # Physical activity
        activity = X[4]
        likelihood *= self.conditional_probs[t]['Activity'].get(activity, 1e-5)
        
        # Health condition
        health = X[5]
        likelihood *= self.conditional_probs[t]['Health'].get(health, 1e-5)
        
        # Income (discretized)
        income_bins = [0, 1700, 3500, float('inf')]
        income_cat = np.digitize([X[6]], income_bins)[0]
        likelihood *= self.conditional_probs[t]['Income'].get(income_cat, 1e-5)
        
        return likelihood
    
    def predict(self, X, threshold=0.5):
        """Make a binary prediction (0 or 1) based on probability threshold"""
        score = self.calculate_score(X)
        if score > 50:
            return 1  # Rule-based recommendation to purchase
            
        # If rule doesn't apply, use Bayesian probability
        prob = self.predict_probability(X)
        return 1 if prob >= threshold else 0

    def display_probability_tables(self):
        """Print the learned probability tables in a readable format"""
        print("\n=== Prior Probabilities P(T) ===")
        print(f"P(T=0): {self.prior_prob[0]:.4f}")
        print(f"P(T=1): {self.prior_prob[1]:.4f}\n")
        
        print("=== Conditional Probabilities P(X|T) ===")
        for t in [0, 1]:
            print(f"\nFor T = {t} ('{'Not Purchase' if t == 0 else 'Purchase'}')")
            for feature, probs in self.conditional_probs[t].items():
                print(f"\n{feature}:")
                for value, prob in probs.items():
                    # Make the display more readable
                    if feature == 'Gender':
                        val_str = "Female" if value == 0 else "Male"
                    elif feature == 'Marital':
                        val_str = "Single" if value == 0 else "Married"
                    elif feature == 'Activity':
                        val_str = ["Sedentary", "Moderately Active", "Active"][value]
                    elif feature == 'Health':
                        val_str = ["Healthy", "Minor Conditions", "Severe Conditions"][value]
                    elif feature == 'Dependents':
                        val_str = f"{value} children" if value < 3 else "3+ children"
                    elif feature == 'Age':
                        age_ranges = ["<30", "30-39", "40-49", "50+"]
                        val_str = age_ranges[value-1] if value <= len(age_ranges) else str(value)
                    elif feature == 'Income':
                        income_ranges = ["<1700", "1700-3499", "3500+"]
                        val_str = income_ranges[value-1] if value <= len(income_ranges) else str(value)
                    else:
                        val_str = str(value)
                    
                    print(f"  {val_str}: {prob:.4f}")

# Example usage
if __name__ == "__main__":
    # Load data (assuming we have lifelnsurance.txt in the same format as described)
    # Format: Gender, Age, Marital, Dependents, Activity, Health, Income, Decision
    try:
        data = np.loadtxt('lifeInsurance.txt')
        X = data[:, :-1]  # Features
        y = data[:, -1]   # Target (decision)
        
        # Train model
        model = LifeInsuranceBayesianModel()
        model.train(X, y)

        model.display_probability_tables()
        
        # Example prediction
        new_person = [1, 29, 1, 3, 2, 2, 1600] 
        decision = model.predict(new_person)
        print(f"Recommended decision: {'Purchase' if decision == 1 else 'Do not purchase'} life insurance")
        
        # Calculate probability
        prob = model.predict_probability(new_person)
        print(f"Probability of purchasing: {prob:.2f}")
        
        # Calculate score
        score = model.calculate_score(new_person)
        print(f"Risk score: {score} points")
        
    except FileNotFoundError:
        print("Error: lifelnsurance.txt not found. Please ensure the data file is in the correct location.")


=== Prior Probabilities P(T) ===
P(T=0): 0.6783
P(T=1): 0.3217

=== Conditional Probabilities P(X|T) ===

For T = 0 ('Not Purchase')

Gender:
  Female: 0.8729
  Male: 0.1271

Age:
  <30: 0.0024
  30-39: 0.2791
  40-49: 0.1626
  50+: 0.5534
  5: 0.0024

Marital:
  Single: 0.5134
  Married: 0.4866

Dependents:
  0 children: 0.2993
  1 children: 0.2774
  2 children: 0.2238
  3+ children: 0.1995

Activity:
  Sedentary: 0.2000
  Moderately Active: 0.5341
  Active: 0.2659

Health:
  Healthy: 0.6244
  Minor Conditions: 0.3244
  Severe Conditions: 0.0512

Income:
  <1700: 0.1022
  1700-3499: 0.7275
  3500+: 0.1679
  4: 0.0024

For T = 1 ('Purchase')

Gender:
  Female: 0.8205
  Male: 0.1795

Age:
  <30: 0.0051
  30-39: 0.1212
  40-49: 0.1162
  50+: 0.7525
  5: 0.0051

Marital:
  Single: 0.3590
  Married: 0.6410

Dependents:
  0 children: 0.1523
  1 children: 0.2030
  2 children: 0.2792
  3+ children: 0.3655

Activity:
  Sedentary: 0.4286
  Moderately Active: 0.5102
  Active: 0.0612

Health:
  